# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moataz-Elzuhery/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



## 1. Signal checks (before I trust anything)

My rule idea leans on two signals. I check each one against real numbers — bucket table, `n` per
bucket, one-word verdict — **before** I code anything into the score. I only use these two
signals to *audit* whether the story holds; `trend_direction` / `trend_pct` are used here as an
outcome check only, never as an input to the scored rule below (per the data dictionary's
leakage warning, they are the label source).

### Signal check 1 — staleness (`days_since_last_update`), behind the refresh flags

**Claim I'm testing:** "The longer since a page was last updated, the more likely it is declining."


In [1]:
import pandas as pd
from pathlib import Path

LOCAL_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
RAW_URL = "https://raw.githubusercontent.com/Moataz-Elzuhery/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(LOCAL_PATH) if LOCAL_PATH.exists() else pd.read_csv(RAW_URL)
print("Rows:", len(df))

bins = [0, 20, 60, 180, 1000]
labels = ["<=20d", "21-60d", "61-180d", "181+d"]
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

signal1 = df.groupby("stale_bucket", observed=True).agg(
    n=("content_id", "size"),
    pct_declining=("trend_direction", lambda s: round((s == "down").mean() * 100, 1)),
    avg_ctr=("ctr", "mean"),
    avg_impressions_90d=("impressions_90d", "mean"),
).round(2)

print(signal1)
print()
print("VERDICT: MIXED")


Rows: 30000
                  n  pct_declining  avg_ctr  avg_impressions_90d
stale_bucket                                                    
<=20d         15866           53.9     0.73              3902.10
21-60d         4742           42.1     0.18              5306.36
61-180d        9218           61.1     0.24              7456.44
181+d           174           47.1     3.69              1172.45

VERDICT: MIXED


**Reading it:** if staleness were doing what I hoped, `pct_declining` would climb steadily as
`stale_bucket` goes from `<=20d` to `181+d`. It doesn't: it dips at `21-60d`, peaks at `61-180d`,
then drops back down at `181+d` — and that last bucket is only **n=174**, a fraction of the
20,000+ pages in the freshest bucket. `avg_ctr` also jumps at `181+d`, which smells like a
different population (older evergreen content that never needed a rewrite) rather than "old and
neglected."

**Verdict: MIXED.** Staleness alone is not a clean, monotonic predictor of decline in this slice.
It's still worth including — some very stale content clearly does need a look — but I won't gate
the rule on staleness alone, and I won't lean on it as the primary driver. This is the
"clearly-explained negative" the task description warns me not to skip: it just saved my rule
from over-trusting one column.

### Signal check 2 — CTR vs. position, behind the CTR-fix logic

**Claim I'm testing:** "CTR should fall as `position_tier` gets worse (top_3 > page_1 > striking >
page_3_5 > deep)."


In [2]:
signal2_raw = df.groupby("position_tier", observed=True).agg(
    n=("content_id", "size"),
    avg_ctr=("ctr", "mean"),
    median_impressions_90d=("impressions_90d", "median"),
).round(2)

print("Raw position_tier table (as shipped):")
print(signal2_raw)
print()

# The data dictionary flags avg_position == 0 as "no data", not rank zero.
# position_tier buckets top_3 as "<= 3", so avg_position == 0 rows silently land in top_3.
top3 = df[df["position_tier"] == "top_3"]
no_data_in_top3 = (top3["avg_position"] == 0).sum()
print(f"Of {len(top3)} rows tagged top_3, {no_data_in_top3} actually have avg_position == 0 (no GSC data).")
print()

real_top3 = top3[top3["avg_position"] > 0]
fake_top3 = top3[top3["avg_position"] == 0]
print("Real top_3 (avg_position 0-3, excl. 0):", "n=", len(real_top3),
      "avg_ctr=", round(real_top3["ctr"].mean(), 2),
      "median impr_90d=", real_top3["impressions_90d"].median())
print("'No data' rows miscoded as top_3:", "n=", len(fake_top3),
      "avg_ctr=", round(fake_top3["ctr"].mean(), 2),
      "median impr_90d=", fake_top3["impressions_90d"].median())
print()
print("VERDICT: MIXED")


Raw position_tier table (as shipped):
                   n  avg_ctr  median_impressions_90d
position_tier                                        
deep            1319     0.15                   218.0
page_1         11814     0.65                  1179.5
page_3_5        7242     0.22                   811.5
striking        7304     0.32                   874.5
top_3           2321     1.48                     3.0

Of 2321 rows tagged top_3, 1205 actually have avg_position == 0 (no GSC data).

Real top_3 (avg_position 0-3, excl. 0): n= 1116 avg_ctr= 2.76 median impr_90d= 53.0
'No data' rows miscoded as top_3: n= 1205 avg_ctr= 0.3 median impr_90d= 1.0

VERDICT: MIXED


## 2. My rule, in plain words — then the ranked queue

**The rule:** *"A page is worth a refresh review if it genuinely has visible search traffic, and
it sits close enough to page 1 that a refresh could plausibly move it — not already top-3
(nothing to fix), not buried past page 5 (a refresh alone won't rescue it), and not one of the
'no data' rows that only look like top-3 because of a data-quality quirk. Staleness on its own
didn't hold up in the signal check, so it acts as a light tie-breaker on ranking, not a hard
gate."*

**Inputs used (all backward-looking, none of them the label or a future window):**
`avg_position`, `position_tier`, `impressions_90d`, `days_since_last_update`.
**Never used as inputs:** `trend_direction`, `trend_pct`, `is_declining_label`, or anything from
`_last_30d` / `_prev_30d` — those either *are* the label source or overlap the label's own window.

**Score:** `score = eligible × impressions_90d × (1 + days_since_last_update / 365)`
**Reason code (one, for every flagged row):** `fixable_position_real_traffic`
**Action label:** `review_for_refresh` if `score > 0`, else `no_action_needed`


In [3]:
import numpy as np

eligible = (
    (df["avg_position"] > 0)
    & (df["position_tier"].isin(["striking", "page_3_5"]))
    & (df["impressions_90d"] >= 300)
)
print("Eligible rows:", eligible.sum(), "of", len(df), f"({eligible.mean()*100:.1f}%)")

work = df.copy()
work["eligible"] = eligible.astype(int)
work["staleness_multiplier"] = 1 + work["days_since_last_update"] / 365
work["score"] = np.where(
    work["eligible"] == 1,
    work["impressions_90d"] * work["staleness_multiplier"],
    0.0,
)
work["reason_code"] = np.where(work["eligible"] == 1, "fixable_position_real_traffic", "not_eligible")
work["action"] = np.where(work["score"] > 0, "review_for_refresh", "no_action_needed")

queue_cols = [
    "content_id", "client_id", "score", "reason_code", "action",
    "position_tier", "avg_position", "impressions_90d", "ctr",
    "days_since_last_update", "content_type",
]
queue = work[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

print()
print("Action counts:")
print(queue["action"].value_counts())
print()
print(queue.head(10).to_string(index=False))


Eligible rows: 10082 of 30000 (33.6%)

Action counts:
action
no_action_needed      19918
review_for_refresh    10082
Name: count, dtype: int64

 rank           content_id         client_id         score                   reason_code             action position_tier  avg_position  impressions_90d  ctr  days_since_last_update    content_type
    1 content_2dba2b1f9536 client_6208ef0f77 569782.317808 fixable_position_real_traffic review_for_refresh      page_3_5          27.9           443434 0.21                     104 keyword article
    2 content_2cb567c3c89b client_6208ef0f77 563181.509589 fixable_position_real_traffic review_for_refresh      page_3_5          22.2           497727 0.10                      48 keyword article
    3 content_b28d1efd668f client_6208ef0f77 368271.649315 fixable_position_real_traffic review_for_refresh      page_3_5          26.2           286608 0.06                     104 keyword article
    4 content_813e88069237 client_6208ef0f77 300109.887671 fixab

In [4]:
from pathlib import Path

out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print("Wrote:", out_path.resolve())
print("Rows written:", len(queue))


Wrote: /outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top10 = queue.head(20)
print(top10.to_string(index=False))


 rank           content_id         client_id         score                   reason_code             action position_tier  avg_position  impressions_90d  ctr  days_since_last_update    content_type
    1 content_2dba2b1f9536 client_6208ef0f77 569782.317808 fixable_position_real_traffic review_for_refresh      page_3_5          27.9           443434 0.21                     104 keyword article
    2 content_2cb567c3c89b client_6208ef0f77 563181.509589 fixable_position_real_traffic review_for_refresh      page_3_5          22.2           497727 0.10                      48 keyword article
    3 content_b28d1efd668f client_6208ef0f77 368271.649315 fixable_position_real_traffic review_for_refresh      page_3_5          26.2           286608 0.06                     104 keyword article
    4 content_813e88069237 client_6208ef0f77 300109.887671 fixable_position_real_traffic review_for_refresh      page_3_5          26.2           233561 0.06                     104 keyword article
    5 cont

## 4. Weak picks + leakage check

**Weak picks, honestly:** 6 of the top 10 (and 8 of the top 20 not shown) belong to a single
client (`client_6208ef0f77`). That's not a bug in the rule — that client genuinely has more
big, page_3_5, high-traffic pages than anyone else in this slice — but a queue that's 60%+ one
client is a bad first list to hand a reviewer: it looks like favoritism, and it buries real
opportunities from smaller clients further down. A fairer version of this rule would rank within
client first (or cap picks per client) before merging into one global queue. I'm not fixing that
here — flagging it is the point of this section.

A second weak spot: rows #2, #7, and #10 were all updated in the last 20-48 days, which
undercuts my own rule's plain-word framing ("worth a refresh because it's stale"). They're in the
queue because of position + traffic, not staleness — which is *consistent* with Signal Check 1
(staleness turned out to be a weak, non-monotonic signal), but it does mean the `reason_code`
`fixable_position_real_traffic` is honest while a human skimming the list might still assume
"stale" when it isn't. Worth a clearer label if this rule goes into a real dashboard.


In [6]:
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label"}
FUTURE_WINDOW = {c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")}

used_inputs = {"avg_position", "position_tier", "impressions_90d", "days_since_last_update"}

assert used_inputs.isdisjoint(FORBIDDEN), "Rule touched a label-derived column!"
assert used_inputs.isdisjoint(FUTURE_WINDOW), "Rule touched a 30d-window column!"

print("Rule inputs:", sorted(used_inputs))
print("Label-derived columns (never used):", sorted(FORBIDDEN))
print("30d-window columns present in data (never used):", sorted(FUTURE_WINDOW))
print()
print("Leakage check passed: no label-derived or 30d-window column feeds the score.")


Rule inputs: ['avg_position', 'days_since_last_update', 'impressions_90d', 'position_tier']
Label-derived columns (never used): ['is_declining_label', 'trend_direction', 'trend_pct']
30d-window columns present in data (never used): ['clicks_last_30d', 'clicks_prev_30d', 'impressions_last_30d', 'impressions_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']

Leakage check passed: no label-derived or 30d-window column feeds the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
